In [ ]:
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, 
    DoubleType, TimestampType, BooleanType
)

# Esquema para silver_documents
schema_silver_documents = StructType([
    StructField("document_id", StringType(), False),
    StructField("file_name", StringType(), False),
    StructField("document_title", StringType(), True),
    StructField("document_category", StringType(), False),
    StructField("language", StringType(), True),
    StructField("clean_text", StringType(), True),
    StructField("num_pages", IntegerType(), True),
    StructField("num_characters", IntegerType(), True),
    StructField("num_words", IntegerType(), True),
    StructField("quality_score", DoubleType(), True),
    StructField("extraction_date", TimestampType(), False)
])

# Esquema para dq_document_quality
schema_dq_quality = StructType([
    StructField("document_id", StringType(), False),
    StructField("file_name", StringType(), False),
    StructField("has_text", BooleanType(), False),
    StructField("num_characters", IntegerType(), True),
    StructField("num_chunks", IntegerType(), True),
    StructField("quality_score", DoubleType(), True),
    StructField("quality_status", StringType(), False), # 'PASSED', 'WARNING', 'FAILED'
    StructField("validation_date", TimestampType(), False)
])

# Crear las tablas Delta en el Lakehouse si no existen
spark.createDataFrame([], schema_silver_documents).write.format("delta").mode("ignore").saveAsTable("silver_documents")
spark.createDataFrame([], schema_dq_quality).write.format("delta").mode("ignore").saveAsTable("dq_document_quality")

print("Tablas 'silver_documents' y 'dq_document_quality' verificadas e inicializadas.")

In [1]:
import os
import re
from datetime import datetime
import pandas as pd
from pyspark.sql import Row
from pyspark.sql.functions import col, expr
from delta.tables import DeltaTable

import pdfplumber
from pypdf import PdfReader
import docx

# ---------------------------------------------------------
# FUNCIONES AUXILIARES DE EXTRACCIÓN Y LIMPIEZA
# ---------------------------------------------------------

def to_local_path(path: str) -> str:
    """Convierte rutas de OneLake/ABFS a rutas locales accesibles por Python en Fabric."""
    if path.startswith("abfss://") or path.startswith("https://"):
        # Extrae todo lo que hay a partir de '/Files/' o '/Tables/'
        if "/Files/" in path:
            relative_path = path.split("/Files/")[1]
            return f"/lakehouse/default/Files/{relative_path}"
        elif "/Tables/" in path:
            relative_path = path.split("/Tables/")[1]
            return f"/lakehouse/default/Tables/{relative_path}"
    return path

def clean_text_content(raw_text: str) -> str:
    """Limpia saltos de línea excesivos, caracteres nulos y espacios múltiples."""
    if not raw_text:
        return ""
    cleaned = re.sub(r'\r\n|\r', '\n', raw_text)
    cleaned = re.sub(r'\n{3,}', '\n\n', cleaned)
    cleaned = re.sub(r'[ \t]{2,}', ' ', cleaned)
    return cleaned.strip()

def extract_from_pdf(file_path: str):
    """Extrae texto de PDF con pdfplumber (fallback a pypdf). Devuelve (texto, num_paginas)."""
    text = ""
    pages_count = 0
    local_path = to_local_path(file_path)
    
    try:
        with pdfplumber.open(local_path) as pdf:
            pages_count = len(pdf.pages)
            for page in pdf.pages:
                page_text = page.extract_text()
                if page_text:
                    text += page_text + "\n"
    except Exception as e:
        # Fallback a PyPDF si pdfplumber falla
        reader = PdfReader(local_path)
        pages_count = len(reader.pages)
        for page in reader.pages:
            t = page.extract_text()
            if t:
                text += t + "\n"
                
    return clean_text_content(text), pages_count

def extract_from_docx(file_path: str):
    """Extrae texto de archivos .docx."""
    local_path = to_local_path(file_path)
    doc = docx.Document(local_path)
    full_text = [para.text for para in doc.paragraphs if para.text]
    return clean_text_content("\n".join(full_text)), 1

def extract_from_excel_csv(file_path: str, file_type: str):
    """Extrae y formatea datos de tablas Excel o CSV a representación en texto."""
    local_path = to_local_path(file_path)
    if file_type == "csv":
        df = pd.read_csv(local_path)
    else:
        df = pd.read_excel(local_path, sheet_name=None)
        if isinstance(df, dict):
            text_sheets = []
            for sheet_name, sheet_df in df.items():
                text_sheets.append(f"--- Hoja: {sheet_name} ---\n" + sheet_df.to_string(index=False))
            return clean_text_content("\n\n".join(text_sheets)), len(df)
            
    return clean_text_content(df.to_string(index=False)), 1

def extract_from_txt(file_path: str):
    """Extrae texto plano."""
    local_path = to_local_path(file_path)
    with open(local_path, "r", encoding="utf-8", errors="ignore") as f:
        content = f.read()
    return clean_text_content(content), 1

# ---------------------------------------------------------
# PROCESAMIENTO PRINCIPAL (CAPA SILVER)
# ---------------------------------------------------------

# Para depurar rápidamente los errores antes de actualizar la tabla:
# Primero reseteamos el estado en Bronze de los 'failed' a 'pending' si necesitas reintentar
spark.sql("UPDATE bronze_documents SET processing_status = 'pending' WHERE processing_status = 'failed'")

bronze_df = spark.table("bronze_documents").filter(col("processing_status") == "pending")
pending_files = bronze_df.collect()

print(f"Archivos listos para procesar en Silver: {len(pending_files)}")

silver_rows = []
dq_rows = []
processed_ids = []
failed_ids = []

existing_silver_ids = set([r.document_id for r in spark.table("silver_documents").select("document_id").collect()])

for file in pending_files:
    doc_id = file.document_id
    file_name = file.file_name
    file_path = file.file_path
    file_type = file.file_type.lower()
    category = file.category
    
    clean_text = ""
    num_pages = 0
    is_corrupt = False
    error_reason = ""
    
    # Validation 1: Documento duplicado en Silver
    if doc_id in existing_silver_ids:
        print(f"[SKIP] Archivo duplicado ya existente en Silver: {file_name}")
        continue

    # Validation 2: Categoria no asignada o 'unknown'
    if not category or category == "unknown":
        category = "operations"
        
    # Extracción de texto según formato
    try:
        if file_type == "pdf":
            clean_text, num_pages = extract_from_pdf(file_path)
        elif file_type in ["docx", "doc"]:
            clean_text, num_pages = extract_from_docx(file_path)
        elif file_type in ["xlsx", "xls", "csv"]:
            clean_text, num_pages = extract_from_excel_csv(file_path, file_type)
        elif file_type in ["txt", "md"]:
            clean_text, num_pages = extract_from_txt(file_path)
        else:
            is_corrupt = True
            error_reason = f"Formato '{file_type}' no soportado."
    except Exception as e:
        is_corrupt = True
        error_reason = f"Error al abrir/extraer archivo (posible corrupción): {str(e)}"
        print(f"[ERROR DETALLADO] {file_name}: {error_reason}") # Imprime el error real en los logs

    num_characters = len(clean_text)
    num_words = len(clean_text.split()) if clean_text else 0
    has_text = num_characters > 0
    
    # ---------------------------------------------------------
    # VALIDACIONES DE CALIDAD DE DATOS ($DQ$) Y SCORE
    # ---------------------------------------------------------
    quality_score = 1.0
    quality_status = "PASSED"
    
    if is_corrupt:
        quality_score = 0.0
        quality_status = "FAILED"
    elif not has_text:
        quality_score = 0.0
        quality_status = "FAILED"
        error_reason = "El documento no contiene texto extraíble."
    elif num_characters < 50:
        quality_score = 0.5
        quality_status = "WARNING"
        error_reason = "Documento demasiado corto (< 50 caracteres)."
        
    num_chunks = max(1, (num_words // 500) + (1 if num_words % 500 > 0 else 0)) if has_text else 0

    if quality_status != "FAILED":
        silver_rows.append(Row(
            document_id=doc_id,
            file_name=file_name,
            document_title=file_name.replace(f".{file_type}", "").replace("_", " ").title(),
            document_category=category,
            language="es",
            clean_text=clean_text,
            num_pages=num_pages,
            num_characters=num_characters,
            num_words=num_words,
            quality_score=quality_score,
            extraction_date=datetime.now()
        ))
        processed_ids.append(doc_id)
    else:
        failed_ids.append(doc_id)

    dq_rows.append(Row(
        document_id=doc_id,
        file_name=file_name,
        has_text=has_text,
        num_characters=num_characters,
        num_chunks=num_chunks,
        quality_score=quality_score,
        quality_status=quality_status,
        validation_date=datetime.now()
    ))

# ---------------------------------------------------------
# PERSISTENCIA Y ACTUALIZACIÓN DE ESTADOS DELTA
# ---------------------------------------------------------
if silver_rows:
    df_silver = spark.createDataFrame(silver_rows, schema=spark.table("silver_documents").schema)
    df_silver.write.format("delta").mode("append").saveAsTable("silver_documents")
    print(f"Insertados {len(silver_rows)} registros en 'silver_documents'.")

if dq_rows:
    df_dq = spark.createDataFrame(dq_rows, schema=spark.table("dq_document_quality").schema)
    df_dq.write.format("delta").mode("append").saveAsTable("dq_document_quality")
    print(f"Registrados {len(dq_rows)} analisis de calidad en 'dq_document_quality'.")

if processed_ids or failed_ids:
    delta_bronze = DeltaTable.forName(spark, "bronze_documents")
    
    if processed_ids:
        delta_bronze.update(
            condition=col("document_id").isin(processed_ids),
            set={"processing_status": expr("'processed'")}
        )
            
    if failed_ids:
        delta_bronze.update(
            condition=col("document_id").isin(failed_ids),
            set={"processing_status": expr("'failed'")}
        )
            
    print("Estados actualizados correctamente en 'bronze_documents'.")

StatementMeta(, f4a69470-90b3-4f60-97fa-2c7b32b374c7, 4, Finished, Available, Finished, False)

Archivos listos para procesar en Silver: 0
